# E011 spline4 deployment parity
Private, internet-disabled, CPU-only deployment notebook. Attach the exact private E011 input dataset version and the ROGII competition data manually. The user is the only authorized notebook runner. This notebook never submits to the competition.


In [ ]:
CONTRACT = {
  "BUNDLE_BYTES": 861617,
  "BUNDLE_FILENAME": "rogii-e011-deployment-inputs-v1.zip.bin",
  "BUNDLE_SHA256": "15a82b9eee1b13d126c9e57b6402c1b307b89238718f0f292eca7773321f66a2",
  "DATASET_REF": "ashok205/rogii-e011-deployment-inputs",
  "DATASET_VERSION": 1,
  "INPUT_RECEIPT_FILENAME": "e011-input-receipt-v1.json",
  "INPUT_RECEIPT_SHA256": "2ce82b05cbf52460af47d52202072c5999d42ed1fa59d3d9c9669f6a23a4d955",
  "MODEL_SHA256": "dd92e8b12f275b6d453e55d6cce2e7c10a43c8a34957d6d47b1b5e5ff5b3c5a8",
  "OUTPUT_MANIFEST_FILENAME": "e011-output-manifest.json",
  "RESULT_ARCHIVE_FILENAME": "rogii-e011-deployment-results-v1.zip",
  "RUN_RECEIPT_FILENAME": "e011-run-receipt.json",
  "SOURCE_COMMIT": "e6fbfaa02e8a72dc3c67713da51fe843417cc114",
  "SUBMISSION_FILENAME": "submission.csv",
  "THREAD_LIMIT": 2,
  "TRAIN_SIGNATURE": "6ebe65b403f80fefd97dcd7bbfce7314252e779a1837c97364cf55761590fe77",
  "TRAIN_WELLS": 773,
  "VISIBLE_FIXTURE": {
    "hidden_rows": 14151,
    "input_files_signature": "f556fa12653ad24ad223a30f4d323153e2e4ac5509143bb4195db467e2ec23fc",
    "known_rows": 5070,
    "rows": 19221,
    "sample_id_order_sha256": "e6a2a380b8751443333064563fe94289055b95a739a3c8ac42d672df28a7e269",
    "sample_rows": 14151,
    "well_ids_sha256": "689e2f6cee0ff0f492f3803b720fc6216739abb0a6355ef60da8e0a6d51c8ff4",
    "wells": 3
  },
  "WELL_DIAGNOSTICS_FILENAME": "e011-well-predictions.json"
}


In [ ]:
import datetime as _dt
import hashlib
import importlib
import json
import os
import platform
import shutil
import sys
import traceback
import zipfile
from pathlib import Path, PurePosixPath

for _name in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
):
    os.environ[_name] = str(CONTRACT["THREAD_LIMIT"])
os.environ["JOBLIB_MULTIPROCESSING"] = "0"

WORKING = Path("/kaggle/working")
RUNTIME_ROOT = WORKING / "rogii-e011-runtime"
PREFLIGHT_PATH = WORKING / "e011-preflight.json"
RUN_RECEIPT_PATH = WORKING / CONTRACT["RUN_RECEIPT_FILENAME"]
SUBMISSION_PATH = WORKING / CONTRACT["SUBMISSION_FILENAME"]
WELL_DIAGNOSTICS_PATH = WORKING / CONTRACT["WELL_DIAGNOSTICS_FILENAME"]


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(16 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def _safe(name: str) -> str:
    raw = str(name).replace("\\", "/")
    raw_parts = raw.split("/")
    value = PurePosixPath(raw)
    if value.is_absolute() or not raw or any(part in {"", ".", ".."} for part in raw_parts) or not value.parts:
        raise RuntimeError(f"unsafe archive member {name!r}")
    return value.as_posix()


def _write_failure(stage: str, exc: BaseException) -> None:
    failure = {
        "schema_version": 1,
        "experiment_id": "E011",
        "status": "FAILED",
        "stage": stage,
        "source_commit": CONTRACT["SOURCE_COMMIT"],
        "failed_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
        "error_type": type(exc).__name__,
        "error": str(exc),
        "traceback": traceback.format_exc(),
        "submission_made": False,
    }
    RUN_RECEIPT_PATH.write_text(json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8")

try:
    bundle_matches = sorted(Path("/kaggle/input").rglob(CONTRACT["BUNDLE_FILENAME"]))
    receipt_matches = sorted(Path("/kaggle/input").rglob(CONTRACT["INPUT_RECEIPT_FILENAME"]))
    if len(bundle_matches) != 1 or len(receipt_matches) != 1:
        raise RuntimeError({
            "bundle_matches": [str(path) for path in bundle_matches],
            "receipt_matches": [str(path) for path in receipt_matches],
        })
    BUNDLE_PATH = bundle_matches[0]
    INPUT_RECEIPT_PATH = receipt_matches[0]
    if BUNDLE_PATH.stat().st_size != int(CONTRACT["BUNDLE_BYTES"]) or _sha256(BUNDLE_PATH) != CONTRACT["BUNDLE_SHA256"]:
        raise RuntimeError("sealed E011 bundle identity mismatch")
    if _sha256(INPUT_RECEIPT_PATH) != CONTRACT["INPUT_RECEIPT_SHA256"]:
        raise RuntimeError("E011 input receipt identity mismatch")
    input_receipt = json.loads(INPUT_RECEIPT_PATH.read_text(encoding="utf-8"))
    if (
        input_receipt["dataset"]["ref"] != CONTRACT["DATASET_REF"]
        or int(input_receipt["dataset"]["version"]) != int(CONTRACT["DATASET_VERSION"])
    ):
        raise RuntimeError("attached E011 dataset ref/version differs from notebook contract")

    if RUNTIME_ROOT.exists():
        shutil.rmtree(RUNTIME_ROOT)
    RUNTIME_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(BUNDLE_PATH) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise RuntimeError("duplicate E011 archive members")
        for name in names:
            _safe(name)
        archive.extractall(RUNTIME_ROOT)
    manifest = json.loads((RUNTIME_ROOT / "bundle_manifest.json").read_text(encoding="utf-8"))
    if manifest["source_commit"] != CONTRACT["SOURCE_COMMIT"] or manifest["model_sha256"] != CONTRACT["MODEL_SHA256"]:
        raise RuntimeError("E011 bundle source/model contract mismatch")
    for entry in manifest["files"]:
        path = RUNTIME_ROOT / _safe(entry["path"])
        if not path.is_file() or path.stat().st_size != int(entry["bytes"]) or _sha256(path) != entry["sha256"]:
            raise RuntimeError(f"E011 extracted member mismatch: {entry['path']}")

    for module_name in tuple(sys.modules):
        if module_name == "rogii_validation" or module_name.startswith("rogii_validation."):
            del sys.modules[module_name]
    importlib.invalidate_caches()
    sys.path.insert(0, str(RUNTIME_ROOT / "src"))
    from rogii_validation.harness import scan_profiles
    from rogii_validation.e011_inference import load_e011_model, profile_e011_test_input, write_e011_submission

    preferred_roots = (
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
    )
    COMPETITION_ROOT = next(
        (
            root
            for root in preferred_roots
            if (root / "train").is_dir() and (root / "test").is_dir() and (root / "sample_submission.csv").is_file()
        ),
        None,
    )
    if COMPETITION_ROOT is None:
        fallback_roots = sorted(
            sample.parent
            for sample in Path("/kaggle/input").rglob("sample_submission.csv")
            if (sample.parent / "train").is_dir() and (sample.parent / "test").is_dir()
        )
        if len(fallback_roots) != 1:
            raise RuntimeError({"competition_roots_found": [str(root) for root in fallback_roots]})
        COMPETITION_ROOT = fallback_roots[0]
    TRAIN_DIR = COMPETITION_ROOT / "train"
    TEST_DIR = COMPETITION_ROOT / "test"
    SAMPLE_SUBMISSION = COMPETITION_ROOT / "sample_submission.csv"
    _, TRAIN_PROFILE = scan_profiles(TRAIN_DIR)
    if int(TRAIN_PROFILE["well_count"]) != int(CONTRACT["TRAIN_WELLS"]) or TRAIN_PROFILE["data_signature"] != CONTRACT["TRAIN_SIGNATURE"]:
        raise RuntimeError({
            "competition_root": str(COMPETITION_ROOT),
            "expected_train_wells": CONTRACT["TRAIN_WELLS"],
            "expected_train_signature": CONTRACT["TRAIN_SIGNATURE"],
            "actual_train_profile": TRAIN_PROFILE,
        })
    TEST_PROFILE = profile_e011_test_input(TEST_DIR, SAMPLE_SUBMISSION)
    if int(TEST_PROFILE["wells"]) == int(CONTRACT["VISIBLE_FIXTURE"]["wells"]):
        fixture = CONTRACT["VISIBLE_FIXTURE"]
        if (
            TEST_PROFILE["input_files_signature"] != fixture["input_files_signature"]
            or TEST_PROFILE["sample_id_order_sha256"] != fixture["sample_id_order_sha256"]
            or int(TEST_PROFILE["sample_rows"]) != int(fixture["sample_rows"])
        ):
            raise RuntimeError("visible E011 competition fixture identity mismatch")
        TEST_MODE = "visible_fixture"
    else:
        TEST_MODE = "private_rerun"
    MODEL_PATH = RUNTIME_ROOT / "experiments/E011/deployment/model.json"
    if _sha256(MODEL_PATH) != CONTRACT["MODEL_SHA256"]:
        raise RuntimeError("E011 deployment model hash mismatch")
    MODEL = load_e011_model(MODEL_PATH)

    PREFLIGHT = {
        "schema_version": 1,
        "experiment_id": "E011",
        "status": "PASS",
        "checked_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
        "source_commit": CONTRACT["SOURCE_COMMIT"],
        "model_sha256": CONTRACT["MODEL_SHA256"],
        "bundle": {"path": str(BUNDLE_PATH), "bytes": BUNDLE_PATH.stat().st_size, "sha256": _sha256(BUNDLE_PATH)},
        "input_receipt": {"path": str(INPUT_RECEIPT_PATH), "sha256": _sha256(INPUT_RECEIPT_PATH)},
        "dataset": {"ref": CONTRACT["DATASET_REF"], "version": CONTRACT["DATASET_VERSION"]},
        "competition_root": str(COMPETITION_ROOT),
        "train_profile": TRAIN_PROFILE,
        "test_mode": TEST_MODE,
        "test_profile": TEST_PROFILE,
        "runtime": {
            "python": sys.version,
            "platform": platform.platform(),
            "logical_cpus": os.cpu_count(),
            "thread_limit": CONTRACT["THREAD_LIMIT"],
            "thread_environment": {
                name: os.environ.get(name)
                for name in (
                    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
                )
            },
            "internet_expected": False,
            "accelerator_expected": "CPU",
        },
        "submission_authorized": False,
    }
    PREFLIGHT_PATH.write_text(json.dumps(PREFLIGHT, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(PREFLIGHT, indent=2, sort_keys=True))
except Exception as exc:
    _write_failure("preflight", exc)
    raise


In [ ]:
import datetime as _dt
import json
import resource
import traceback

import numpy as np
import sklearn
from threadpoolctl import threadpool_info, threadpool_limits

STARTED_AT = _dt.datetime.now(_dt.timezone.utc)
try:
    with threadpool_limits(limits=CONTRACT["THREAD_LIMIT"]):
        THREAD_POOLS_BEFORE = threadpool_info()
        if any(int(item.get("num_threads", 0) or 0) > int(CONTRACT["THREAD_LIMIT"]) for item in THREAD_POOLS_BEFORE):
            raise RuntimeError({"thread_pool_limit_failed": THREAD_POOLS_BEFORE})
        RESULT = write_e011_submission(MODEL, TEST_DIR, SAMPLE_SUBMISSION, SUBMISSION_PATH)
        WELL_DIAGNOSTICS_PATH.write_text(
            json.dumps(RESULT["well_predictions"], indent=2, sort_keys=True, allow_nan=False) + "\n",
            encoding="utf-8",
        )
        THREAD_POOLS_AFTER = threadpool_info()
        if any(int(item.get("num_threads", 0) or 0) > int(CONTRACT["THREAD_LIMIT"]) for item in THREAD_POOLS_AFTER):
            raise RuntimeError({"thread_pool_limit_failed_after": THREAD_POOLS_AFTER})
    ENDED_AT = _dt.datetime.now(_dt.timezone.utc)
    print(json.dumps({
        "status": "PASS",
        "test_mode": TEST_MODE,
        "wells": RESULT["wells"],
        "rows": RESULT["rows"],
        "submission_sha256": RESULT["sha256"],
        "wall_seconds": (ENDED_AT - STARTED_AT).total_seconds(),
        "max_rss_kb": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss),
    }, indent=2, sort_keys=True))
except Exception as exc:
    _write_failure("inference", exc)
    raise


In [ ]:
import hashlib
import json
import resource
import shutil
import zipfile
from pathlib import Path, PurePosixPath


def _output_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(16 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def _output_safe(name: str) -> str:
    raw = str(name).replace("\\", "/")
    parts = raw.split("/")
    value = PurePosixPath(raw)
    if value.is_absolute() or not raw or any(part in {"", ".", ".."} for part in parts) or not value.parts:
        raise RuntimeError(f"unsafe output member {name!r}")
    return value.as_posix()

try:
    output_sources = [SUBMISSION_PATH, WELL_DIAGNOSTICS_PATH, PREFLIGHT_PATH]
    if not all(path.is_file() for path in output_sources):
        raise RuntimeError("E011 expected output file is missing before packaging")
    manifest = {
        "schema_version": 1,
        "experiment_id": "E011",
        "source_commit": CONTRACT["SOURCE_COMMIT"],
        "model_sha256": CONTRACT["MODEL_SHA256"],
        "input_bundle_sha256": CONTRACT["BUNDLE_SHA256"],
        "files": [
            {"path": path.name, "bytes": path.stat().st_size, "sha256": _output_sha256(path)}
            for path in output_sources
        ],
    }
    MANIFEST_PATH = WORKING / CONTRACT["OUTPUT_MANIFEST_FILENAME"]
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    ARCHIVE_PATH = WORKING / CONTRACT["RESULT_ARCHIVE_FILENAME"]
    temporary = ARCHIVE_PATH.with_suffix(".zip.tmp")
    temporary.unlink(missing_ok=True)
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in [*output_sources, MANIFEST_PATH]:
            info = zipfile.ZipInfo(_output_safe(path.name), date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.create_system = 3
            info.external_attr = 0o100644 << 16
            archive.writestr(info, path.read_bytes())
    temporary.replace(ARCHIVE_PATH)
    RECEIPT = {
        "schema_version": 1,
        "experiment_id": "E011",
        "status": "COMPLETE",
        "source_commit": CONTRACT["SOURCE_COMMIT"],
        "model_sha256": CONTRACT["MODEL_SHA256"],
        "input_bundle": {"sha256": CONTRACT["BUNDLE_SHA256"], "bytes": CONTRACT["BUNDLE_BYTES"]},
        "dataset": {"ref": CONTRACT["DATASET_REF"], "version": CONTRACT["DATASET_VERSION"]},
        "competition_root": str(COMPETITION_ROOT),
        "train_profile": TRAIN_PROFILE,
        "test_mode": TEST_MODE,
        "test_profile": TEST_PROFILE,
        "started_at_utc": STARTED_AT.isoformat(),
        "ended_at_utc": ENDED_AT.isoformat(),
        "wall_seconds": (ENDED_AT - STARTED_AT).total_seconds(),
        "max_rss_kb": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "thread_limit": CONTRACT["THREAD_LIMIT"],
        "thread_pools_before": THREAD_POOLS_BEFORE,
        "thread_pools_after": THREAD_POOLS_AFTER,
        "internet_expected": False,
        "accelerator_expected": "CPU",
        "prediction": {key: value for key, value in RESULT.items() if key != "well_predictions"},
        "output_manifest": {"filename": MANIFEST_PATH.name, "bytes": MANIFEST_PATH.stat().st_size, "sha256": _output_sha256(MANIFEST_PATH)},
        "result_archive": {"filename": ARCHIVE_PATH.name, "bytes": ARCHIVE_PATH.stat().st_size, "sha256": _output_sha256(ARCHIVE_PATH)},
        "submission_file_created": True,
        "kaggle_submission_made": False,
    }
    RUN_RECEIPT_PATH.write_text(json.dumps(RECEIPT, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
    print(json.dumps(RECEIPT, indent=2, sort_keys=True))
except Exception as exc:
    _write_failure("packaging", exc)
    raise
